# Engineering Assignment E2-M: ICU Vitals Alarm Detector

**Engineering 10 · Module 2 · Fall 2026 · George Witmer**

**Level:** Mastery

---

> **About the three levels of E2.** This assignment exists in three independent versions — Novice (E2-N), Intermediate (E2-I), and Mastery (E2-M). Each is a different problem of similar conceptual difficulty; the levels differ mainly in scope and how long they take. You may submit one, two, or all three — each is graded on its own. **You do not need to complete a lower level before attempting a higher one.**

---

## Scenario

You are a biomedical engineer working on the firmware for a bedside patient monitor in an intensive care unit. The monitor records a heart-rate (HR) and blood-oxygen-saturation (SpO₂) reading once per minute and writes one line per minute to a text log. Briefly elevated heart rate or briefly low SpO₂ are normal — patients move, talk, cough — and triggering an alarm on every one-minute blip would just train the nurses to ignore the monitor. Your job is to scan a log and flag the **sustained** events: stretches during which **both** the heart rate was too high *and* the SpO₂ was too low for at least three minutes in a row. Each sustained event is one alarm.

## Learning Objectives

After completing this level, you will be able to:
1. Clean and parse loosely-formatted text using `.strip()`, `.split()`, `.lower()`, and `.replace()`.
2. Build `datetime` objects from text timestamps using `datetime.strptime()` and format them back with `strftime`-style codes inside an f-string.
3. Use a list comprehension to turn a list of raw log lines into a list of structured records.
4. Use a **nested `for` loop** to scan for a multi-step pattern (here: a sliding window of N consecutive flagged minutes).
5. Produce a clean, column-aligned report using f-string field widths.

## Prerequisites

You should be comfortable with the following topics from Module 1 and this module:
- `lists-basics`, `list-comprehensions` — building lists from other lists
- `for-loops`, `nested-loops` — `for` with `range`, `enumerate`, and a loop inside a loop
- `fstrings` — field widths, decimal places, and `strftime` codes inside an f-string (e.g. `f"{t:%H:%M:%S}"`)
- `strings-methods` — `.strip()`, `.split()`, `.lower()`, `.replace()`
- `datetime-basics` — `datetime`, `datetime.strptime()`, `timedelta`
- `functions-defining`, `branching`, `boolean-logic` — `def`, `if`, comparison operators (Module 1)

*If any of these feel shaky, review the topic notes before starting — this assignment assumes them.*

## Background

**Heart rate (HR)**, measured in **beats per minute (bpm)**, is how many times the heart contracts in a minute. A resting adult is normally 60–100 bpm; sustained values above 100 bpm define a condition called **tachycardia**, which can indicate stress, fever, blood loss, or arrhythmia. In an ICU setting, an HR above about 100 bpm for more than a couple of minutes is worth a clinician's attention.

**Blood oxygen saturation**, written **SpO₂**, is the percentage of hemoglobin molecules in the patient's arterial blood that are carrying oxygen. It is measured non-invasively with a finger-clip pulse oximeter. A healthy SpO₂ at sea level is 95–100%. Values below about 92% are called **hypoxemia** and mean the lungs and circulation aren't keeping the tissues adequately oxygenated; below about 88% the patient is in trouble.

> For this assignment, the alarm criteria are: **HR > 100 bpm AND SpO₂ < 92%**, sustained for **3 or more consecutive minutes**. Both conditions must hold *together* in the same minute, and the run must be at least three minutes long. A 2-minute bout, or a 4-minute bout in which only HR was elevated, does not trigger.

**Why "sustained" matters.** Real patient-monitor designs care deeply about **alarm fatigue**: when alarms go off constantly for benign reasons, clinicians stop responding promptly, and people die. Requiring a sustained threshold crossing — rather than a single-minute trigger — is a standard, very simple kind of *alarm de-bouncing*. You will see much more sophisticated versions of the same idea in any real signal-processing course.

No calculus is needed; the only "math" is counting consecutive minutes that meet two simple inequalities.

**For further reading:**
- [American Heart Association: Tachycardia](https://www.heart.org/en/health-topics/arrhythmia/about-arrhythmia/tachycardia--fast-heart-rate)
- [The Joint Commission — Sentinel Event Alert on alarm safety](https://www.jointcommission.org/) (search for *alarm fatigue*)
- [Python `datetime` module documentation](https://docs.python.org/3/library/datetime.html)

*You do not need to read these to complete the assignment — they are here if you want to go deeper.*

## Problem Statement

You are given 21 lines from a one-minute-resolution bedside monitor log spanning 08:00–08:20 on 2026-09-14. Each line has three pipe-separated fields: a timestamp, an `HR=<bpm>` field, and a `SpO2=<percent>` field. Some lines have stray whitespace and inconsistent capitalization — your parser must handle that. Detect every sustained alarm (HR > 100 AND SpO₂ < 92 for ≥ 3 consecutive minutes) and print a report.

Use **this exact dataset** at the top of your notebook:

```python
from datetime import datetime

log_lines = [
    "2026-09-14 08:00:00 | HR=78 | SpO2=98",
    "2026-09-14 08:01:00 | HR=82 | SpO2=97",
    " 2026-09-14 08:02:00 | HR=85 | SpO2=97 ",
    "2026-09-14 08:03:00 | HR=110 | SpO2=92",
    "2026-09-14 08:04:00 | hr=115 | spo2=91",
    "2026-09-14 08:05:00 | HR=122 | SpO2=90",
    "2026-09-14 08:06:00 | HR=125 | SpO2=89",
    "2026-09-14 08:07:00 | HR=95  | SpO2=95",
    "2026-09-14 08:08:00 | HR=88  | SpO2=96",
    "2026-09-14 08:09:00 | HR=85  | SpO2=97",
    "2026-09-14 08:10:00 | HR=82  | SpO2=97",
    "2026-09-14 08:11:00 | HR=105 | SpO2=94",
    "2026-09-14 08:12:00 | HR=108 | SpO2=95",
    "2026-09-14 08:13:00 | HR=80  | SpO2=96",
    "2026-09-14 08:14:00 | HR=78  | SpO2=97",
    "2026-09-14 08:15:00 | HR=76  | SpO2=98",
    "2026-09-14 08:16:00 | HR=130 | SpO2=88",
    "2026-09-14 08:17:00 | HR=128 | SpO2=87",
    "2026-09-14 08:18:00 | HR=135 | SpO2=86",
    "2026-09-14 08:19:00 | HR=140 | SpO2=85",
    "2026-09-14 08:20:00 | HR=92  | SpO2=94",
]

HR_HIGH    = 100   # bpm
SPO2_LOW   = 92    # percent
WINDOW_MIN = 3     # minutes
```

Note the messy bits in the dataset: line 3 has stray spaces at both ends, line 5 has lowercase `hr=` and `spo2=`, and several lines have a double space before the pipe after the HR field. Your parser must cope with all of this without special-casing — using `.strip()`, `.lower()`, and `.replace()` will do it cleanly.

### Requirements

Your code must:
- Copy the dataset and the three constants above into your notebook exactly as shown.
- Define a function — for example, `parse_line(line)` — that takes one raw log string and returns a 3-element list `[timestamp, hr, spo2]` where `timestamp` is a `datetime` object (built with `datetime.strptime`) and the other two are `int`s. Inside the function use `.strip()`, `.split("|")`, `.lower()`, and `.replace()` to clean the fields. The function must handle the messy lines correctly — do not special-case any particular line.
- Use a **list comprehension** to build a list called `records` by calling your parse function on every entry in `log_lines`.
- Build a list `flags` of length 21 where `flags[i]` is `True` when minute `i` satisfies *both* `hr > HR_HIGH` *and* `spo2 < SPO2_LOW`, and `False` otherwise. A list comprehension is the natural way to do this — but a plain `for` loop is fine.
- Detect each **sustained alarm**. Use a **nested `for` loop**: the outer loop walks each possible window start `i` from `0` to `len(flags) - WINDOW_MIN`; the inner loop checks whether all `WINDOW_MIN` flags starting at `i` are `True`. Only record `i` as an alarm **start** when (a) the window check passes and (b) the previous minute was *not* in alarm (so that you do not record the same alarm three times when a 5-minute bout slides past three valid start positions). For the very first minute (`i == 0`), there is no previous minute, so condition (b) is satisfied automatically.
- For each alarm start, find how long the run continues — keep extending while `flags[end]` is still `True`. Record the start timestamp, end timestamp, and duration in minutes.
- Print the report exactly in the format below: the header three lines, a blank line, then either `No alarms in this window.` or a numbered table of alarms followed by a `Total alarms: N` line. Use `strftime` codes inside f-strings to format timestamps as `HH:MM:SS`.

### Expected Output

When you run your program with the dataset above, you should see exactly:

```
ICU Vitals Alarm Report
Source: 21 minutes of data from 2026-09-14 08:00 to 08:20
Alarm criteria: HR > 100 bpm AND SpO2 < 92% sustained 3 min or longer

  #  Start       End           Duration
------------------------------------------
  1  08:04:00    08:06:00          3 min
  2  08:16:00    08:19:00          4 min

Total alarms: 2
```

*A note on why the first alarm starts at 08:04 and not 08:03: at 08:03 the SpO₂ is 92, and our criterion is **strictly less than** 92. This is a deliberate part of the problem — make sure your inequality matches the spec.*

## Your Solution

Write your code in the cell below. You may add additional cells (code or markdown) as needed to organize your work or explain your thinking.

In [17]:
# Your solution here
from datetime import datetime

log_lines = [
    "2026-09-14 08:00:00 | HR=78 | SpO2=98",
    "2026-09-14 08:01:00 | HR=82 | SpO2=97",
    " 2026-09-14 08:02:00 | HR=85 | SpO2=97 ",
    "2026-09-14 08:03:00 | HR=110 | SpO2=92",
    "2026-09-14 08:04:00 | hr=115 | spo2=91",
    "2026-09-14 08:05:00 | HR=122 | SpO2=90",
    "2026-09-14 08:06:00 | HR=125 | SpO2=89",
    "2026-09-14 08:07:00 | HR=95  | SpO2=95",
    "2026-09-14 08:08:00 | HR=88  | SpO2=96",
    "2026-09-14 08:09:00 | HR=85  | SpO2=97",
    "2026-09-14 08:10:00 | HR=82  | SpO2=97",
    "2026-09-14 08:11:00 | HR=105 | SpO2=94",
    "2026-09-14 08:12:00 | HR=108 | SpO2=95",
    "2026-09-14 08:13:00 | HR=80  | SpO2=96",
    "2026-09-14 08:14:00 | HR=78  | SpO2=97",
    "2026-09-14 08:15:00 | HR=76  | SpO2=98",
    "2026-09-14 08:16:00 | HR=130 | SpO2=88",
    "2026-09-14 08:17:00 | HR=128 | SpO2=87",
    "2026-09-14 08:18:00 | HR=135 | SpO2=86",
    "2026-09-14 08:19:00 | HR=140 | SpO2=85",
    "2026-09-14 08:20:00 | HR=92  | SpO2=94",
]

HR_HIGH    = 100   # bpm
SPO2_LOW   = 92    # percent
WINDOW_MIN = 3     # minutes



def parse_line (line):
    parts = line.split("|") #split string at each |
    timeS = parts[0].strip() #get rid of leading and tailing spaces
    timestamp = datetime.strptime(timeS, "%Y-%m-%d %H:%M:%S")

    hr_str = parts[1].lower().replace("hr=", "").strip()
    hr = int(hr_str)

    spo2_str = parts[2].lower().replace("spo2=", "").strip()
    spo2 = int(spo2_str)

    return [timestamp, hr, spo2]



record = [parse_line(line) for line in log_lines] #sets record equal to values returned by calling parse on each line of data

flags = [hr > HR_HIGH and spo2 < SPO2_LOW for timestamp, hr, spo2 in record] #all instances where conditions for alarm are met, not organized by time yet


alarms = []

for i in range(len(flags) - WINDOW_MIN +1):
    window_valid = all(flags[i : i + WINDOW_MIN]) # see if next 3 minutes are in "alarm" state
    
    prev_not_alarm = (i == 0) or (not flags[i - 1]) # see if first min or last min was not an alarm to set multiple off
    
    if window_valid and prev_not_alarm:
        
        endI = i + WINDOW_MIN - 1
        
        while endI + 1 < len(flags) and flags[endI + 1]: #alarm cont past 3 min
            endI += 1

        start_time = record[i][0]
        end_time = record[endI][0]
        duration = endI - i + 1

        alarms.append((start_time, end_time, duration))
        
print("ICU Vitals Alarm Report")
print("Source: 21 minutes of data from 2026-09-14 08:00 to 08:20")
print("Alarm criteria: HR > 100 bpm AND SpO2 < 92% sustained 3 min or longer \n")

if not alarms:
    print("No alarms in this window.") #empty list its false
    
else:
    print(f"{'#':<3} {'Start':<10} {'End':<10} {'Duration'}")
    print("-" * 37)
    
    for i, (start, end, duration) in enumerate(alarms, 1): #string formatting for datetime
        
        start_fmt = start.strftime("%H:%M:%S") #
        end_fmt = end.strftime("%H:%M:%S")
        print(f"{i:<3} {start_fmt:<10} {end_fmt:<10} {duration} min")


print(f"\nTotal alarms: {len(alarms)}")


ICU Vitals Alarm Report
Source: 21 minutes of data from 2026-09-14 08:00 to 08:20
Alarm criteria: HR > 100 bpm AND SpO2 < 92% sustained 3 min or longer 

#   Start      End        Duration
-------------------------------------
1   08:04:00   08:06:00   3 min
2   08:16:00   08:19:00   4 min

Total alarms: 2


---

## Preparing for the Oral Exam

Your oral exam will follow a fixed rubric the TA works through with you. Use the four sections below to prepare — points are awarded only when you can do all four.

### 1. Files Submitted

Before the oral exam can begin, you must show the TA that your notebook has been uploaded to Course Site. Do this *before* the appointment, not during it.

### 2. Demonstration

You will run your code in front of the TA. It must produce the expected output with no errors and no warnings, and detect exactly the two alarms shown. Your parser must really call `.strip()`/`.lower()`/`.replace()` and `datetime.strptime()` on each line, and your nested loop must actually do the sliding-window check — printing the alarm table as a hardcoded literal will fail this step.

### 3. Explanation (no looking at your code)

The TA will ask you to describe, **without looking at your code or these notes**, what the problem is and how your solution approaches it. Practice this out loud before the exam. You should be able to say something like *"I wrote a parse function that strips whitespace, splits on the pipe, lowercases each field, and pulls the numeric part out of `hr=` and `spo2=` to build a datetime and two ints. I run that function over every log line with a list comprehension. Then I make a parallel list of booleans, true for any minute where HR and SpO₂ both exceed the thresholds. The alarm detector is a nested for loop: the outer loop picks a window start, the inner loop checks the next three flags, and I only record an alarm when all three are true and the previous minute wasn't in alarm — that prevents counting the same bout multiple times. For each start I extend forward to find the duration, then print the report with formatted timestamps."*

### 4. Code Review

With your code in front of you, you will walk the TA through it. Be ready to explain:
- What each line or function does and why it is there.
- Specific decisions — for example, *what `.split("|")` returns when the input is `"a|b|c"` versus `"a||c"`*, *why you need `.lower()` before `.replace("hr=", "")` (and what happens to line 5 if you don't)*, *the exact `strptime` format string you used and what `%Y-%m-%d %H:%M:%S` means*, *what your inner loop's exit condition is and what would happen if you forgot the "previous minute not in alarm" check*, *why the outer loop range is `n - WINDOW_MIN + 1` and not just `n`*.
- How your code would behave on a variation the TA may propose — for example, *change the sustained threshold to 5 minutes and confirm both bouts disappear*, *add a third bout that is exactly 2 minutes long and confirm it is correctly ignored*, *also report the **peak HR** observed during each alarm (this requires a second pass over the run — write it on the spot)*. You may be asked to edit and re-run on the spot.

You will **not** have access to the internet, AI tools, or chat assistants during the oral exam. If you cannot explain your own code, you will not pass — regardless of how well it runs.

---

## How to Submit

1. Save this notebook with your work as `E2-M_LastnameFirstname.ipynb`.
2. Upload to Course Site under this assignment.
3. Request an oral exam from a TA **during office hours** (Fritz 402). Oral exams are not given during lab sections.
4. Points are awarded only after you pass the oral exam. If you do not pass on the first try, the TA will tell you what to revise and you may try again later — but it must still be before the module deadline.

**Deadline:** End of Module 2 — see the course schedule. **No extensions.**

---

## Academic Honesty

You may use the internet, including generative AI, to **help you learn**. You may not copy another student's work, and you may not submit code you cannot explain. The oral exam is where this gets checked: if you cannot explain your own code without electronic help, you will not pass — regardless of how good the code is.

See the [course academic honesty policy](https://www.lehigh.edu/~ejk0/cheating.html).